# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DataType, DoubleType, IntegerType
from pyspark.sql.functions import trim, col
from pyspark.sql.functions import split, col
from pyspark.sql.window import Window

#Reading From Bronze Table

# Data Transformations

In [0]:
df_sales = spark.table("dev_project.bronze.crm_sales_details")
df_sales.printSchema()
df_sales.limit(5).display()

## Sales Fact Table

### Reading From Bronze

In [0]:
df_sales = spark.table("dev_project.bronze.crm_sales_details")

### Trim String Columns

In [0]:
string_cols = [f.name for f in df_sales.schema.fields if isinstance(f.dataType, StringType)]

for col_name in string_cols:
    df_sales = df_sales.withColumn(
        col_name,
        F.when(F.trim(F.col(col_name)) == "", None)
         .otherwise(F.trim(F.col(col_name)))
    )

### Deduplication

In [0]:
# Natural key is (sls_ord_num, sls_prd_key) — one order can have multiple line items
window_dedup = Window.partitionBy("sls_ord_num", "sls_prd_key") \
                     .orderBy(F.col("_ingestion_ts").desc())

df_sales = df_sales.withColumn("_row_num", F.row_number().over(window_dedup)) \
                   .filter(F.col("_row_num") == 1) \
                   .drop("_row_num")

### Date Casting from Int to Date

In [0]:
# try_to_date() returns NULL for invalid values like 0 instead of throwing an error
df_sales = df_sales \
    .withColumn("sls_order_dt", F.try_to_date(F.col("sls_order_dt").cast("string"), F.lit("yyyyMMdd"))) \
    .withColumn("sls_ship_dt",  F.try_to_date(F.col("sls_ship_dt").cast("string"),  F.lit("yyyyMMdd"))) \
    .withColumn("sls_due_dt",   F.try_to_date(F.col("sls_due_dt").cast("string"),   F.lit("yyyyMMdd")))

### Invalid Date Nulling

In [0]:
# Null out ship_date and due_date where they are before order_date — source system bug
df_sales = df_sales \
    .withColumn(
        "sls_ship_dt",
        F.when(F.col("sls_ship_dt") < F.col("sls_order_dt"), None)
         .otherwise(F.col("sls_ship_dt"))
    ) \
    .withColumn(
        "sls_due_dt",
        F.when(F.col("sls_due_dt") < F.col("sls_order_dt"), None)
         .otherwise(F.col("sls_due_dt"))
    )

###  Sales Validation + Price/Quantity Cleanup

In [0]:
# Null out non-positive price and quantity — these are invalid
df_sales = df_sales \
    .withColumn(
        "sls_price",
        F.when(F.col("sls_price") <= 0, None)
         .otherwise(F.col("sls_price"))
    ) \
    .withColumn(
        "sls_quantity",
        F.when(F.col("sls_quantity") <= 0, None)
         .otherwise(F.col("sls_quantity"))
    )

# Recalculate sls_sales where it is null, zero, negative,
# or doesn't reconcile with quantity * price
df_sales = df_sales.withColumn(
    "sls_sales",
    F.when(
        F.col("sls_sales").isNull() |
        (F.col("sls_sales") <= 0) |
        (F.col("sls_sales") != F.col("sls_quantity") * F.col("sls_price")),
        F.col("sls_quantity") * F.col("sls_price")
    ).otherwise(F.col("sls_sales"))
)

# Derive sls_price from sls_sales / sls_quantity where sls_price is null
# Source has 12 rows with valid sales and quantity but missing price
df_sales = df_sales.withColumn(
    "sls_price",
    F.when(
        F.col("sls_price").isNull() &
        F.col("sls_sales").isNotNull() &
        F.col("sls_quantity").isNotNull(),
        (F.col("sls_sales") / F.col("sls_quantity")).cast("integer")
    ).otherwise(F.col("sls_price"))
)

### Rename + Final Select

In [0]:
RENAME_MAP_SALES = {
    "sls_ord_num":  "order_number",
    "sls_prd_key":  "product_key",
    "sls_cust_id":  "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt":  "ship_date",
    "sls_due_dt":   "due_date",
    "sls_sales":    "sales_amount",
    "sls_quantity": "quantity",
    "sls_price":    "unit_price"
}

for old_col, new_col in RENAME_MAP_SALES.items():
    df_sales = df_sales.withColumnRenamed(old_col, new_col)

# Drop Bronze metadata columns
df_sales_silver = df_sales.select(
    "order_number",
    "product_key",
    "customer_id",
    "order_date",
    "ship_date",
    "due_date",
    "sales_amount",
    "quantity",
    "unit_price"
)

### Cell — Validation

In [0]:
print("\n" + "="*60)
print(" SALES DATA QUALITY VALIDATION")
print("="*60)
print(f"Total records:          {df_sales_silver.count():,}")

dup_check = df_sales_silver.groupBy("order_number", "product_key").count().filter("count > 1")
print(f"Duplicate keys:         {dup_check.count()}")
if dup_check.count() > 0:
    dup_check.show()

null_orders = df_sales_silver.filter(F.col("order_number").isNull()).count()
print(f"Null order numbers:     {null_orders}")

null_customers = df_sales_silver.filter(F.col("customer_id").isNull()).count()
print(f"Null customer IDs:      {null_customers}")

invalid_dates = df_sales_silver.filter(
    F.col("ship_date").isNotNull() & (F.col("ship_date") < F.col("order_date"))
).count()
print(f"Invalid ship dates:     {invalid_dates}")

invalid_sales = df_sales_silver.filter(
    F.col("sales_amount").isNull() | (F.col("sales_amount") <= 0)
).count()
print(f"Invalid sales amounts:  {invalid_sales}")

null_prices = df_sales_silver.filter(F.col("unit_price").isNull()).count()
print(f"Null unit prices:       {null_prices}")
print("="*60 + "\n")

In [0]:
# Derive unit_price from sales_amount / quantity where unit_price is null
# All 12 affected rows have valid sales_amount and quantity so division is safe
df_sales_silver = df_sales_silver.withColumn(
    "unit_price",
    F.when(
        F.col("unit_price").isNull() &
        F.col("sales_amount").isNotNull() &
        F.col("quantity").isNotNull(),
        (F.col("sales_amount") / F.col("quantity")).cast("integer")
    ).otherwise(F.col("unit_price"))
)

### Writting Into Silver Layer

In [0]:
TARGET_TABLE = "dev_project.silver.crm_sales_details"

# --- FIRST RUN: Table doesn't exist yet — write and exit ---
if not spark.catalog.tableExists(TARGET_TABLE):
    df_sales_silver.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(TARGET_TABLE)
    print("Initial load complete — table created.")

else:
    # Load existing Silver to find already-ingested order+product combinations
    df_existing_keys = spark.table(TARGET_TABLE) \
        .select("order_number", "product_key")

    # Only insert rows that don't already exist in Silver
    df_new = df_sales_silver.join(
        df_existing_keys,
        on=["order_number", "product_key"],
        how="left_anti"
    )

    if not df_new.isEmpty():
        df_new.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(TARGET_TABLE)
        print(f"Incremental load complete — {df_new.count():,} new records inserted.")
    else:
        print("No new records to insert.")